In [1]:
import os
import random
import csv

# Path to your dataset
dataset_path = "/mnt/abka03/xlvlm_data/coco_cc0_plus/val"
output_csv = "mcq_dataset.csv"

# Get all class names
classes = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

data = []

for cls in classes:
    class_path = os.path.join(dataset_path, cls)
    for img_file in os.listdir(class_path):
        img_path = os.path.join(class_path, img_file)
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            continue

        # Ensure unique wrong options
        wrong_options = random.sample([c for c in classes if c != cls], 3)
        options = wrong_options + [cls]
        random.shuffle(options)

        # Remove underscores from each option
        options = [opt.replace('_', '') for opt in options]

        question = "What object is in this image?"
        data.append({
            "image_path": img_path,
            "question": question,
            "options": options,
            "ground_truth": cls.replace('_', '')
        })

# Write to CSV
with open(output_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image_path", "question", "options", "ground_truth"])
    writer.writeheader()
    for row in data:
        # Convert options list to a string for CSV
        row["options"] = ", ".join(row["options"])
        writer.writerow(row)

print(f"MCQ-I dataset saved to {output_csv}")


MCQ-I dataset saved to mcq_dataset.csv


In [2]:
import os
import json
import requests
from collections import defaultdict

# CONFIG
ANNOTATION_DIR = "/mnt/abka03/raw_data_download/coco_cc0/annotations"
OUTPUT_DIR = "coco_cc0_plus"
IMAGES_PER_CATEGORY = 3

def get_annotation_file(split):
    return os.path.join(ANNOTATION_DIR, f"instances_{split}2017.json")

def get_caption_file(split):
    return os.path.join(ANNOTATION_DIR, f"captions_{split}2017.json")

def load_coco_data(annotation_file, caption_file):
    with open(annotation_file, "r") as f:
        data = json.load(f)
    with open(caption_file, "r") as f:
        captions_data = json.load(f)

    # Filter licenses: CC0 or ID > 5
    commercial_licenses = {lic["id"] for lic in data["licenses"] if lic["id"] > 5 or "CC0" in lic["name"]}
    print("Using licenses:", commercial_licenses)

    # Filter images by license
    images_by_id = {img["id"]: img for img in data["images"] if img["license"] in commercial_licenses}

    categories = {cat["id"]: cat["name"] for cat in data["categories"]}
    category_images = defaultdict(set)
    annotations_by_image = defaultdict(list)

    for ann in data["annotations"]:
        if ann["image_id"] in images_by_id:
            category_images[ann["category_id"]].add(ann["image_id"])
            annotations_by_image[ann["image_id"]].append(ann)

    captions_by_image = defaultdict(list)
    for ann in captions_data["annotations"]:
        if ann["image_id"] in images_by_id:
            captions_by_image[ann["image_id"]].append(ann["caption"])

    return images_by_id, categories, category_images, annotations_by_image, captions_by_image

def download_image(url, save_path):
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            with open(save_path, "wb") as f:
                f.write(r.content)
            return True
    except Exception as e:
        print(f"Download failed: {e}")
    return False

def process_split(split_name, annotation_dir, output_dir):
    annotation_file = get_annotation_file(split_name)
    caption_file = get_caption_file(split_name)
    print(f"Processing {split_name} split...")

    images_by_id, categories, category_images, annotations_by_image, captions_by_image = load_coco_data(annotation_file, caption_file)

    for cat_id, img_ids in category_images.items():
        cat_name = categories[cat_id]
        folder_path = os.path.join(output_dir, split_name, cat_name)
        os.makedirs(folder_path, exist_ok=True)

        downloaded = 0
        for img_id in img_ids:
            if downloaded >= IMAGES_PER_CATEGORY:
                break
            img_info = images_by_id[img_id]
            filename = os.path.join(folder_path, f"{img_id}.jpg")
            meta_file = os.path.join(folder_path, f"{img_id}.json")

            if download_image(img_info["coco_url"], filename):
                downloaded += 1
                metadata = {
                    "image_id": img_id,
                    "file_name": img_info["file_name"],
                    "captions": captions_by_image[img_id],
                    "annotations": annotations_by_image[img_id]
                }
                with open(meta_file, "w") as f:
                    json.dump(metadata, f, indent=2)
                print(f"[{split_name}/{cat_name}] Downloaded {downloaded}/{IMAGES_PER_CATEGORY} (with captions & segments)")

# --- Run for train and val splits ---
for split in ["train", "val"]:
    process_split(split, ANNOTATION_DIR, OUTPUT_DIR)

print(f"Done! Images with captions & segmentation saved in: {OUTPUT_DIR}")

Processing train split...


KeyboardInterrupt: 

In [ ]:
annotation_path = "/mnt/abka03/Projects/xl-vlms/notebooks/coco_cc0_plus/val/fork/181796.json"
image_path = "/mnt/abka03/Projects/xl-vlms/notebooks/coco_cc0_plus/val/fork/181796.jpg"

In [ ]:
import json
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Load category map from COCO annotation file
coco_ann_path = "/mnt/abka03/raw_data_download/coco_cc0/annotations/instances_val2017.json"
with open(coco_ann_path, "r") as f:
    coco_data = json.load(f)
category_map = {str(cat["id"]): cat["name"] for cat in coco_data["categories"]}

# Load image and annotation
annotation_path = "/mnt/abka03/Projects/xl-vlms/notebooks/coco_cc0_plus/val/orange/138115.json"
image_path = "/mnt/abka03/Projects/xl-vlms/notebooks/coco_cc0_plus/val/orange/138115.jpg"

with open(annotation_path, "r") as f:
    annotation = json.load(f)
img = Image.open(image_path)

fig, ax = plt.subplots(1, figsize=(8, 8))
ax.imshow(img)

# Color palette for annotations
colors = plt.cm.get_cmap('tab10', len(annotation.get("annotations", [])))

# Show bounding boxes and category names
for idx, ann in enumerate(annotation.get("annotations", [])):
    color = colors(idx) if len(annotation.get("annotations", [])) > 1 else (1,0,0,0.4)
    bbox = ann.get('bbox', None)
    category_id = ann.get('category_id', None)
    category_name = category_map.get(str(category_id), str(category_id)) if category_id is not None else 'Unknown'
    if bbox:
        x, y, w, h = bbox
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y, category_name, fontsize=10, color=color, backgroundcolor='white', weight='bold')

plt.title(f"Image ID: {annotation.get('image_id', '')}")
plt.axis('off')
plt.show()

In [ ]:
# Install Lang-Segment-Anything (only needs to run once per environment)
!pip install -U git+https://github.com/luca-medeiros/lang-segment-anything.git
# Optional: pillow, numpy, matplotlib often present in notebooks
# !pip install pillow numpy matplotlib


In [ ]:
from typing import List, Union, Optional, Sequence
from PIL import Image
from lang_sam import LangSAM
import numpy as np

try:
    import torch  # optional, for dtype/array handling
except Exception:
    torch = None


def load_langsam(device: Optional[str] = None, **kwargs) -> LangSAM:
    """
    Create and return a LangSAM model instance.

    Args:
        device: Optional device spec (e.g., "cuda", "cpu"). If unsupported, it's ignored.
        **kwargs: Forwarded to LangSAM initializer (kept for future flexibility).

    Returns:
        LangSAM model instance.
    """
    # LangSAM currently doesn't expose a device arg directly in some versions; keep API future-proof.
    _ = device  # placeholder, in case future versions support device
    return LangSAM(**kwargs)


def _to_pil(img: Union[str, Image.Image, np.ndarray]) -> Image.Image:
    if isinstance(img, Image.Image):
        return img.convert("RGB")
    if isinstance(img, str):
        return Image.open(img).convert("RGB")
    if isinstance(img, np.ndarray):
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        return Image.fromarray(img.astype(np.uint8)).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(img)}")


def _to_numpy(a) -> np.ndarray:
    """Safely convert tensor/array/list to numpy array."""
    if torch is not None and hasattr(a, "detach") and hasattr(a, "cpu"):
        return a.detach().cpu().numpy()
    return np.array(a)


def _binarize_mask(mask: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    if mask.dtype == bool:
        return mask
    if np.issubdtype(mask.dtype, np.floating):
        return mask > threshold
    return mask > 0


def _clamp_bbox_xywh(x: float, y: float, w: float, h: float, W: int, H: int) -> List[int]:
    # Convert to inclusive right/bottom to clamp safely, then back to width/height
    x1 = x + w
    y1 = y + h
    x0 = float(np.clip(x, 0, max(W - 1, 0)))
    y0 = float(np.clip(y, 0, max(H - 1, 0)))
    xr = float(np.clip(x1, 0, W))
    yb = float(np.clip(y1, 0, H))

    left = int(np.floor(x0))
    top = int(np.floor(y0))
    right = int(np.ceil(xr))
    bottom = int(np.ceil(yb))

    # Ensure minimum size of 1x1
    if right <= left:
        right = min(W, left + 1)
    if bottom <= top:
        bottom = min(H, top + 1)

    return [left, top, right - left, bottom - top]


def _bboxes_from_masks(masks: Union[np.ndarray, Sequence], image_size: tuple, threshold: float = 0.5) -> List[List[int]]:
    """
    Compute tight XYWH bboxes for each mask.

    Args:
        masks: 2D mask, stack of masks (N,H,W), list/tuple of masks, or tensor equivalents.
        image_size: (W, H) of the source image.
        threshold: Threshold for binarizing float masks.

    Returns:
        List of [x, y, w, h] ints, one per non-empty mask.
    """
    W, H = image_size

    # Normalize to a list of 2D numpy arrays
    masks_list: List[np.ndarray] = []
    if isinstance(masks, (list, tuple)):
        for m in masks:
            m_np = _to_numpy(m)
            if m_np.ndim > 2:
                m_np = np.squeeze(m_np)
            masks_list.append(m_np)
    else:
        m_np = _to_numpy(masks)
        if m_np.ndim == 3:  # (N,H,W)
            for k in range(m_np.shape[0]):
                masks_list.append(m_np[k])
        else:  # (H,W)
            masks_list.append(np.squeeze(m_np))

    bboxes: List[List[int]] = []
    for m in masks_list:
        m_bin = _binarize_mask(m, threshold)
        if not m_bin.any():
            continue
        ys, xs = np.where(m_bin)
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()
        x = float(x_min)
        y = float(y_min)
        w = float(x_max - x_min + 1)
        h = float(y_max - y_min + 1)
        bboxes.append(_clamp_bbox_xywh(x, y, w, h, W, H))
    return bboxes


def _bboxes_from_xywh_boxes(boxes: Union[np.ndarray, Sequence], image_size: tuple) -> List[List[int]]:
    """Normalize incoming boxes to integer [x,y,w,h] clamped to image bounds."""
    W, H = image_size
    arr = _to_numpy(boxes)
    arr = np.asarray(arr).reshape(-1, 4)
    out: List[List[int]] = []
    for x, y, w, h in arr:
        out.append(_clamp_bbox_xywh(float(x), float(y), float(w), float(h), W, H))
    return out


def _extract_bboxes_from_result(result: dict, image_size: tuple, prefer_masks: bool = True) -> List[List[int]]:
    boxes = result.get("boxes")
    masks = result.get("masks")
    if masks is None:
        masks = result.get("mask")

    if prefer_masks and masks is not None:
        bbs = _bboxes_from_masks(masks, image_size)
        if bbs:  # only fall back if masks empty
            return bbs

    if boxes is not None:
        return _bboxes_from_xywh_boxes(boxes, image_size)

    return []


def predict_bboxes_for_tag(
    model: LangSAM,
    images: Sequence[Union[str, Image.Image, np.ndarray]],
    tag: str,
    prefer_masks: bool = True,
) -> List[List[List[int]]]:
    """
    Predict bounding boxes for a tag over a list of images.

    Args:
        model: LangSAM model instance.
        images: Sequence of image paths, PIL Images, or numpy arrays.
        tag: Text prompt for the object name (e.g., "apple").
        prefer_masks: If True, compute tight boxes from masks; otherwise rely on model boxes.

    Returns:
        A list of length len(images). Each element is a list of [x, y, w, h] integer bboxes
        for the corresponding image; can be empty if nothing is detected.
    """
    images_pil = [_to_pil(im) for im in images]

    all_bboxes: List[List[List[int]]] = []
    for img in images_pil:
        # Predict for a single image to keep API compatibility across versions
        results = model.predict([img], [tag])
        # Results is expected to be a list (per image). Gather all detections for this image.
        img_bbs: List[List[int]] = []
        if isinstance(results, dict):
            results = [results]
        for res in results:
            img_bbs.extend(_extract_bboxes_from_result(res, img.size, prefer_masks))
        all_bboxes.append(img_bbs)

    return all_bboxes


def predict_bboxes_for_tag_batched(
    model: LangSAM,
    images: Sequence[Union[str, Image.Image, np.ndarray]],
    tag: str,
    prefer_masks: bool = True,
    batch_size: int = 8,
) -> List[List[List[int]]]:
    """
    Batched version of predict_bboxes_for_tag for faster throughput.

    Processes images in chunks (batch_size) and calls model.predict on each chunk.

    Returns: Same as predict_bboxes_for_tag.
    """
    images_pil = [_to_pil(im) for im in images]
    N = len(images_pil)
    all_bboxes: List[List[List[int]]] = [[] for _ in range(N)]

    def chunked(seq, n):
        for i in range(0, len(seq), n):
            yield i, seq[i : i + n]

    for start, imgs_chunk in chunked(images_pil, batch_size):
        tags_chunk = [tag] * len(imgs_chunk)
        try:
            results = model.predict(imgs_chunk, tags_chunk)
        except Exception:
            # Fallback to per-image if batch call not supported
            for idx, im in enumerate(imgs_chunk, start=start):
                single_res = model.predict([im], [tag])
                img_bbs: List[List[int]] = []
                if isinstance(single_res, dict):
                    single_res = [single_res]
                for res in single_res:
                    img_bbs.extend(_extract_bboxes_from_result(res, im.size, prefer_masks))
                all_bboxes[idx] = img_bbs
            continue

        # Parse batch results. Common case: list of length == len(imgs_chunk)
        if isinstance(results, list) and len(results) == len(imgs_chunk):
            for off, (im, res_i) in enumerate(zip(imgs_chunk, results)):
                img_bbs: List[List[int]] = []
                if isinstance(res_i, dict):
                    img_bbs.extend(_extract_bboxes_from_result(res_i, im.size, prefer_masks))
                elif isinstance(res_i, (list, tuple)):
                    for r in res_i:
                        if isinstance(r, dict):
                            img_bbs.extend(_extract_bboxes_from_result(r, im.size, prefer_masks))
                all_bboxes[start + off] = img_bbs
        else:
            # Unexpected shape; fallback per image for this chunk
            for idx, im in enumerate(imgs_chunk, start=start):
                single_res = model.predict([im], [tag])
                img_bbs: List[List[int]] = []
                if isinstance(single_res, dict):
                    single_res = [single_res]
                for res in single_res:
                    img_bbs.extend(_extract_bboxes_from_result(res, im.size, prefer_masks))
                all_bboxes[idx] = img_bbs

    return all_bboxes


# Example (commented):
# model = load_langsam()
# tag = "apple"
# image_list = [
#     "/mnt/abka03/Projects/xl-vlms/data/train/apple/417834.jpg",
#     "/mnt/abka03/Projects/xl-vlms/data/val/apple/307734.jpg",
# ]
# bboxes_per_image = predict_bboxes_for_tag_batched(model, image_list, tag, batch_size=8)
# print(bboxes_per_image)  # [[ [x,y,w,h], ...], [ ... ]]

### LangSAM utilities for object bounding boxes

This notebook now provides:
- `load_langsam()` to load the model once.
- `predict_bboxes_for_tag(model, images, tag)` to return a list (per image) of `[x, y, w, h]` boxes.

Each item in the returned list corresponds to one input image and can contain multiple boxes if multiple objects are detected in that image.

In [ ]:
# Load the LangSAM model once for reuse in this notebook
model = load_langsam()
print(type(model).__name__, "loaded")


In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

apple_dir = "/mnt/abka03/Projects/xl-vlms/data/val/apple"

def _list_images(folder):
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    files = []
    for n in sorted(os.listdir(folder)):
        p = os.path.join(folder, n)
        if os.path.isfile(p) and os.path.splitext(p)[1].lower() in exts:
            files.append(p)
    return files

image_paths = _list_images(apple_dir)
print(f"Found {len(image_paths)} images in {apple_dir}")

# Run LangSAM for tag 'apple' in batches
batch_size = 8  # adjust based on memory
boxes_per_image = predict_bboxes_for_tag_batched(model, image_paths, tag="apple", batch_size=batch_size)

# Visualize crops per image
for img_path, bbs in zip(image_paths, boxes_per_image):
    img = Image.open(img_path).convert("RGB")
    print(f"\n{os.path.basename(img_path)} -> {len(bbs)} detections")

    if not bbs:
        continue

    cols = min(5, max(1, len(bbs)))
    rows = (len(bbs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    if rows * cols == 1:
        axes = [axes]
    elif rows == 1:
        axes = list(axes)
    else:
        axes = [ax for row in axes for ax in row]

    for ax, (x, y, w, h) in zip(axes, bbs):
        crop = img.crop((x, y, x + w, y + h))
        ax.imshow(crop)
        ax.set_title(f"{x},{y},{w},{h}")
        ax.axis("off")

    # Hide any extra subplots
    for ax in axes[len(bbs):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
!pip uninstall -U 

In [ ]:
!pip install qwen-vl-utils

In [ ]:
!conda install -c conda-forge openjdk

#### Demo: using src/langsam_utils module

Below we import the reusable utilities and run the same apple-folder detection using the module API.

In [ ]:
import sys, os
from PIL import Image
import matplotlib.pyplot as plt

# Ensure project root is importable (adjust if notebook moved)
proj_root = "/mnt/abka03/Projects/xl-vlms"
if proj_root not in sys.path:
    sys.path.append(proj_root)

from src.langsam_utils import load_langsam as load_langsam_ext, predict_bboxes_for_tag_batched as predict_bboxes_for_tag_batched_ext

# Load model from the module, independent from the earlier notebook-defined functions
model_ext = load_langsam_ext()

apple_dir = "/mnt/abka03/Projects/xl-vlms/data/val/apple"

def _list_images(folder):
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    return [
        os.path.join(folder, n)
        for n in sorted(os.listdir(folder))
        if os.path.isfile(os.path.join(folder, n)) and os.path.splitext(n)[1].lower() in exts
    ]

image_paths = _list_images(apple_dir)
print(f"[utils demo] Found {len(image_paths)} images in {apple_dir}")

# Use the external utils for batched detection
boxes_per_image_ext = predict_bboxes_for_tag_batched_ext(model_ext, image_paths, tag="apple", batch_size=8)

# Quick visualization (same style)
for img_path, bbs in zip(image_paths, boxes_per_image_ext):
    img = Image.open(img_path).convert("RGB")
    print(f"\n[utils demo] {os.path.basename(img_path)} -> {len(bbs)} detections")

    if not bbs:
        continue

    cols = min(5, max(1, len(bbs)))
    rows = (len(bbs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    if rows * cols == 1:
        axes = [axes]
    elif rows == 1:
        axes = list(axes)
    else:
        axes = [ax for row in axes for ax in row]

    for ax, (x, y, w, h) in zip(axes, bbs):
        crop = img.crop((x, y, x + w, y + h))
        ax.imshow(crop)
        ax.set_title(f"{x},{y},{w},{h}")
        ax.axis("off")

    for ax in axes[len(bbs):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()